In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages
from dotenv import load_dotenv

from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool

import requests
import random
import os

ModuleNotFoundError: No module named 'langchain_community'

In [ ]:
load_dotenv()

In [ ]:
llm = ChatOpenAI(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("Grok_Api_key"),
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
search_tool = DuckDuckGoSearchRun(region='us-en')

# Calculator tool
@tool
def calculator(first_num: float, second_num : float, operation: str) -> dict:
    """ Perform a basic arithmetic operation on two number.
    Supported operation: add, sub, mul, div
    """
    try:
        if operation == 'add':
            result = first_num + second_num
        elif operation == 'sub':
            result = first_num - second_num
        elif operation == 'mul':
            result = first_num * second_num
        elif operation == 'div':
            if second_num == 0:
                return {"error":"Division by zero is not allowed"}
            result = first_num / second_num
        else:
            return {"error": f"Unsupported operation '{operation}'"}            
        
        return {"First_num": first_num, "Second_num" : second_num, "Operation": operation, 'result': result}
    except Exception as e:
        return {"error": str(e)}


# Custom stock price find tool
@tool
def get_stock_price(symbol:str) -> dict:
    """ Fetch latest stock price for a given symbol (e.g "AAPL" ,"TSLA")
    using Alpha Vantage with Api key in the URL.
    """
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey=7M9TA0NSWVASV591"
    r = requests.get(url)
    return r.json()

In [ ]:
# make tool list
tools = [get_stock_price, search_tool, calculator]

# Make the llm tool-aware
llm_with_tools = llm.bind_tools(tools)

In [ ]:
# State
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]

In [ ]:
def chat_node(state:ChatState): # call the chat node
    'LLm node that may answer or request a tool call.'
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {'messages': [response]}

tool_node = ToolNode(tools) # Tool node call

In [ ]:
#Graph 
graph = StateGraph(ChatState)

graph.add_node("chat_node",chat_node)
graph.add_node("tools_node", tool_node)

In [ ]:
#Edge
graph.add_edge(START, 'chat_node')

graph.add_edge('chat_node', tools_condition)

graph.add_edge('tools_node','chat_node' )

In [ ]:
chatbot = graph.compile()
chatbot